In [6]:
# import pandas as pd
# import glob
# import os
# import io
# # 指定CSV文件所在的文件夹路径
# csv_folder_path = r"/mnt/c/csv/"
# # 指定Parquet文件将要保存的文件夹路径
# parquet_folder_path = r"/mnt/c/parquet/"

# # 使用glob模块获取所有CSV文件的路径
# csv_files = glob.glob(os.path.join(csv_folder_path, '*.csv'))

# # 遍历所有CSV文件
# for csv_file in csv_files:
#     # 读取CSV文件
#     df = pd.read_csv(csv_file)
#     df = df.astype(str).replace({'nan': None})
#     file_name = os.path.basename(csv_file)
#     parquet_file = os.path.join(parquet_folder_path, file_name.replace('.csv', '.parquet'))
    
#     # 写入Parquet文件
#     df.to_parquet(parquet_file, engine='pyarrow')
#     print(f"Converted {file_name} to Parquet format.")

# print("All files have been converted to Parquet format.")


In [2]:
from pyspark.sql import SparkSession

# 强制停止旧 Session，释放 JVM 句柄
if 'spark' in locals():
    spark.stop()

spark = (SparkSession.builder
    .appName("Phil_IP_Bridge_POC")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    
    # --- 使用物理 IP 避开 localhost 陷阱 ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://172.22.19.65:9000") 
    
    # Iceberg Catalog 配置
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://bc2-standardised-restricted-ide/warehouse")
    
    # MinIO 认证
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    
    # --- 核心优化：禁止死循环重试 ---
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "5000")
    .config("spark.hadoop.fs.s3a.retry.limit", "1")  # 只重试一次
    .config("spark.hadoop.fs.s3a.retry.interval", "1s")
    .getOrCreate())

print("🚀 IP 桥接 Session 已成功初始化！")

# --- 验证 ---
try:
    spark.sql("SHOW DATABASES IN local").show()
    print("✅ 数据库列表拉取成功！")
except Exception as e:
    print(f"❌ 还是不行，请尝试在浏览器访问 http://172.22.19.65:9000 查看是否有 XML 输出。")
    print(f"具体报错：{e}")

your 131072x1 screen size is bogus. expect trouble
26/04/01 14:33:31 WARN Utils: Your hostname, DESKTOP-CDCLH86 resolves to a loopback address: 127.0.1.1; using 172.22.19.65 instead (on interface eth0)
26/04/01 14:33:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/phil/ldp/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/phil/.ivy2/cache
The jars for the packages stored in: /home/phil/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c83bfb46-14b2-468c-b062-d9ab6aa48d9c;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 225ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.

🚀 IP 桥接 Session 已成功初始化！


26/04/01 14:33:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------------+
|     namespace|
+--------------+
|bc2_cde_config|
|  bc2_cde_genl|
|  bc2_cde_rstk|
+--------------+

✅ 数据库列表拉取成功！


In [3]:
# 确保在正确的 Catalog 下

databases = [
    "local.bc2_cde_config",
    "local.bc2_cde_genl",
    "local.bc2_cde_rstk"
]

for db in databases:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")

In [3]:
# spark.sql("DROP DATABASE IF EXISTS local.c2_cde_rstk CASCADE");



In [4]:
spark.sql("SHOW DATABASES IN local").show()

+--------------+
|     namespace|
+--------------+
|bc2_cde_config|
|  bc2_cde_genl|
|  bc2_cde_rstk|
+--------------+



In [5]:
spark.sql("USE local")
spark.sql("CREATE DATABASE IF NOT EXISTS bc2_cde_rstk")
spark.sql("USE bc2_cde_rstk")

DataFrame[]

In [6]:
#007.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
# 执行建表 DDL
spark.sql("""CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF(
    RTN_RESP_EV_ID                          VARCHAR(100), 
    AI_ENT_CD                               VARCHAR(100), 
    RPT_POS_DT                              DATE, 
    SBMN_DTTM                               TIMESTAMP, 
    HALF_YR_END_DT                          DATE          ,
    APPR_MONY_BRKR_NM                       VARCHAR(2000) , 
    TNGBL_ASSET_AMT                         DECIMAL(30,10), 
    INV_AMT                                 DECIMAL(30,10), 
    OTH_NON_CUR_ASSET_NM                    VARCHAR(2000) , 
    OTH_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    TOT_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    CASH_AT_BANK_AND_IN_HAND_AMT            DECIMAL(30,10), 
    DEBTOR_AMT                              DECIMAL(30,10), 
    OTH_CUR_ASSET_NM                        VARCHAR(2000) , 
    OTH_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_ASSET_AMT                           DECIMAL(30,10), 
    CROR_AMT                                DECIMAL(30,10), 
    LN_AMT                                  DECIMAL(30,10), 
    DFR_TAX_AMT                             DECIMAL(30,10), 
    OTH_CUR_LIAB_NM                         VARCHAR(2000) , 
    OTH_CUR_LIAB_AMT                        DECIMAL(30,10), 
    TOT_CUR_LIAB_AMT                        DECIMAL(30,10), 
    LONG_TERM_LN_AMT                        DECIMAL(30,10), 
    OTH_LONG_TERM_LIAB_NM                   VARCHAR(2000) , 
    OTH_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LIAB_AMT                            DECIMAL(30,10), 
    NET_ASSET_AMT                           DECIMAL(30,10), 
    PD_UP_SHR_CAP_AMT                       DECIMAL(30,10), 
    SHR_PREM_ACCT_AMT                       DECIMAL(30,10), 
    REVALQ_RESV_AMT                         DECIMAL(30,10), 
    OTH_RESV_NM                             VARCHAR(2000) , 
    OTH_RESV_AMT                            DECIMAL(30,10), 
    PRFT_AND_LOSS_ACCT_AMT                  DECIMAL(30,10), 
    TOT_SHRHLD_FUND_AMT                     DECIMAL(30,10), 
    ROW_VLD_STS_CD                          VARCHAR(255)  , 
    ROW_VLD_MSG_TXT                         VARCHAR(2000) ,
    TX_DT                                   DATE, 
    INSE_DTTM                               TIMESTAMP,       
    UPDT_DTTM                               TIMESTAMP       
) USING iceberg 
TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！")

26/04/01 14:34:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！


In [7]:

#008.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
# 执行建表 DDL
spark.sql("""CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN(
    RTN_RESP_EV_ID                                                      VARCHAR(100),
    AI_ENT_CD                                                           VARCHAR(100),
    RPT_POS_DT                                                          DATE,
    SBMN_DTTM                                                           TIMESTAMP,
    HALF_YR_END_DT                                                      DATE,
    APPR_MONY_BRKR_NM                                                   VARCHAR(2000),
    TRAN_BSS_REVN_GEN_IND                                               VARCHAR(100),
    TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                 DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                           DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                                DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                          VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                         DECIMAL(30,10),
    FX_DRTV_FX_SWAP_AMT                                                 DECIMAL(30,10),
    FX_DRTV_NDF_AMT                                                     DECIMAL(30,10),
    FX_DRTV_NDO_AMT                                                     DECIMAL(30,10),
    FX_DRTV_VANILLA_FX_OPT_AMT                                          DECIMAL(30,10),
    FX_DRTV_OTH_FX_DRTV_NM                                              VARCHAR(2000),
    FX_DRTV_OTH_FX_DRTV_AMT                                             DECIMAL(30,10),
    INT_RT_DRTV_SNGL_CURY_IRS_AMT                                       DECIMAL(30,10),
    INT_RT_DRTV_CROSS_CURY_IRS_AMT                                      DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                    DECIMAL(30,10),
    INT_RT_DRTV_FRA_AMT                                                 DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_OPT_AMT                                          DECIMAL(30,10),
    INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                      VARCHAR(2000),
    INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                     DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_I_NM                                          VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_I_AMT                                         DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_II_NM                                         VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_II_AMT                                        DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_III_NM                                        VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_III_AMT                                       DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN        DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                                VARCHAR(2000),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                               DECIMAL(30,10),
    TOT_HK_RELVNT_BUSN_TRAN_AMT                                         DECIMAL(30,10),
    ROW_VLD_STS_CD                                                      VARCHAR(255),
    ROW_VLD_MSG_TXT                                                     VARCHAR(2000),
    TX_DT                                                               DATE,
    INSE_DTTM                                                           TIMESTAMP,
    UPDT_DTTM                                                           TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);


""")

print("表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！")

表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！


In [8]:

#009.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
# 执行建表 DDL
spark.sql("""
CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN(
    SURV_RESP_EV_ID                                                     VARCHAR(100),
    AI_ENT_CD                                                           VARCHAR(100),
    RPT_POS_DT                                                          DATE,
    SBMN_DTTM                                                           TIMESTAMP,
    HALF_YR_END_DT                                                      DATE,
    APPR_MONY_BRKR_NM                                                   VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                 DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                           DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                                DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                          VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                         DECIMAL(30,10),
    FX_DRTV_FX_SWAP_AMT                                                 DECIMAL(30,10),
    FX_DRTV_NDF_AMT                                                     DECIMAL(30,10),
    FX_DRTV_NDO_AMT                                                     DECIMAL(30,10),
    FX_DRTV_VANILLA_FX_OPT_AMT                                          DECIMAL(30,10),
    FX_DRTV_OTH_FX_DRTV_NM                                              VARCHAR(2000),
    FX_DRTV_OTH_FX_DRTV_AMT                                             DECIMAL(30,10),
    INT_RT_DRTV_SNGL_CURY_IRS_AMT                                       DECIMAL(30,10),
    INT_RT_DRTV_CROSS_CURY_IRS_AMT                                      DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                    DECIMAL(30,10),
    INT_RT_DRTV_FRA_AMT                                                 DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_OPT_AMT                                          DECIMAL(30,10),
    INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                      VARCHAR(2000),
    INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                     DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_I_NM                                          VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_I_AMT                                         DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_II_NM                                         VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_II_AMT                                        DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_III_NM                                        VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_III_AMT                                       DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN        DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                                VARCHAR(2000),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                               DECIMAL(30,10),
    TOT_HK_RELVNT_BUSN_TRAN_AMT                                         DECIMAL(30,10),
    ROW_VLD_STS_CD                                                      VARCHAR(255),
    ROW_VLD_MSG_TXT                                                     VARCHAR(2000),
    TX_DT                                                               DATE,
    INSE_DTTM                                                           TIMESTAMP,
    UPDT_DTTM                                                           TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN 已成功建立！")

表： AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN 已成功建立！


In [ ]:
#DROP TABLE [IF EXISTS] table_name;

# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF;")
# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN;")
# spark.sql("TRUNCATE TABLE local.bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN;")



DataFrame[]

In [37]:
#Check Table status before start ETL
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF""").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN""").show()
print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN""").show()



AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------+---------+----------+---------+--------------+-----------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+--------------+---------------+-----+---------+---------+
|RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_C

In [11]:
#Check parquets before start ETL
print("data of iib")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()
print("data of iv")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()
print("data of iii")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251203_AMB_20251203000001_Passed_20251203.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251203_AMB_20251204000001_Passed_20251204.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_CHC101_20251205_AMB_20251205000001_Passed_20251205.parquet`;""").show()




data of iib
+--------------------+--------------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+-------------------+-----------------+---------+----------+--------------+
|      ROW_VLD_STS_CD|     ROW_VLD_MSG_TXT|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_CUR_LIAB_NM|OTH_CUR_LIAB_AMT|TOT_CUR_LIAB_AMT|LONG_TERM_LN_AMT|OTH_LONG_TERM_LIAB_NM|OTH_LONG

ETL Start：

In [78]:
#切换文件
#file_name = "CHC101_20251203_AMB_20251203000001_Passed_20251203"
# file_name = "CHC101_20251203_AMB_20251204000001_Passed_20251204"
file_name = "CHC101_20251205_AMB_20251205000001_Passed_20251205"

In [79]:

# 2. 构建 SQL 语句
sql_query_crtbl_iib = f"""
CREATE OR REPLACE TEMPORARY VIEW temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half
USING PARQUET OPTIONS (
  path 's3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_{file_name}.parquet'
);
"""
sql_query_crtbl_iv = f"""
CREATE OR REPLACE TEMPORARY VIEW temp_view_amb_part_iv_revn_from_hk_relvnt_busn
USING PARQUET OPTIONS (
  path 's3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_{file_name}.parquet'
);
"""
sql_query_crtbl_iii = f"""
CREATE OR REPLACE TEMPORARY VIEW temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn
USING PARQUET OPTIONS (
  path 's3a://bc2-raw-restricted-ide/20260330/bc2_amb-part-iii-tran-amount-of-hk-relvnt-busn_1_{file_name}.parquet'
);
"""


In [80]:

# 3. 执行 SQL 语句
spark.sql(sql_query_crtbl_iib)
spark.sql(sql_query_crtbl_iv)
spark.sql(sql_query_crtbl_iii)


DataFrame[]

In [81]:
spark.sql("select * from temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half").show()
spark.sql("select * from temp_view_amb_part_iv_revn_from_hk_relvnt_busn").show()
spark.sql("select * from temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn").show()

+--------------------+--------------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+-------------------+-----------------+---------+----------+--------------+
|      ROW_VLD_STS_CD|     ROW_VLD_MSG_TXT|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_CUR_LIAB_NM|OTH_CUR_LIAB_AMT|TOT_CUR_LIAB_AMT|LONG_TERM_LN_AMT|OTH_LONG_TERM_LIAB_NM|OTH_LONG_TERM_LIAB_A

In [82]:
#check data before del:
print("check data before del:")
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN").show()
print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()


check data before del:
AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------------+---------+----------+-------------------+--------------+-----------------+---------------+------------+--------------------+---------------------+---------------------+----------------------------+-------------+----------------+-----------------+-----------------+-------------+-------------+-------------+-------------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+--------------+---------------+-----------------+-----------------+---------------+-----------+-------------+----------------------+-------------------+--------------------+--------------------+-----+--------------------+--------------------+
|      RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|          SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|     INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_

In [83]:
#检查满足删除条件的数据

print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("""SELECT * FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
""").show()

print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("""SELECT * FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iv_revn_from_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
""").show()

print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("""SELECT * FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
""").show()


AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------+---------+----------+---------+--------------+-----------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+--------------+---------------+-----+---------+---------+
|RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_C

In [84]:
#  构建 SQL 语句
sql_query_dl_iib = f"""DELETE FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
"""

sql_query_dl_iv = f"""DELETE FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iv_revn_from_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
"""

sql_query_dl_iii = f"""DELETE FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t1
WHERE EXISTS (
    SELECT 1 
    FROM temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn t2
    WHERE t1.AI_ENT_CD = t2.AI_ENT_CD
        AND t1.RPT_POS_DT = TO_DATE(t2.RPT_POS_DT, 'yyyyMMdd')
        AND t1.SBMN_DTTM <= TO_TIMESTAMP(t2.SBMN_DTTM, 'yyyyMMddHHmmss')
);
"""

In [85]:

#  执行 SQL 语句
spark.sql(sql_query_dl_iib)
spark.sql(sql_query_dl_iv)
spark.sql(sql_query_dl_iii)

DataFrame[]

In [86]:
#check data after del:
print("check data after del:")
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN").show()
print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()

check data after del:
AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------------+---------+----------+-------------------+--------------+-----------------+---------------+------------+--------------------+---------------------+---------------------+----------------------------+-------------+----------------+-----------------+-----------------+-------------+-------------+-------------+-------------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+--------------+---------------+-----------------+-----------------+---------------+-----------+-------------+----------------------+-------------------+--------------------+--------------------+-----+--------------------+--------------------+
|      RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|          SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|     INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_H

In [87]:
#检查满足插入条件的数据
spark.sql("""SELECT * FROM temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
""").show()

spark.sql("""SELECT * FROM temp_view_amb_part_iv_revn_from_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
""").show()

spark.sql("""SELECT * FROM temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
""").show()


+--------------------+--------------------+---------------+-------+--------------------+---------------------+---------------------+----------------------------+----------+----------------+-----------------+-----------------+-------------+--------+------+-----------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+------------+-------------+-----------------+-----------------+---------------+-----------+------------+----------------------+-------------------+-------------------+-----------------+---------+----------+--------------+
|      ROW_VLD_STS_CD|     ROW_VLD_MSG_TXT|TNGBL_ASSET_AMT|INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAND_AMT|DEBTOR_AMT|OTH_CUR_ASSET_NM|OTH_CUR_ASSET_AMT|TOT_CUR_ASSET_AMT|TOT_ASSET_AMT|CROR_AMT|LN_AMT|DFR_TAX_AMT|OTH_CUR_LIAB_NM|OTH_CUR_LIAB_AMT|TOT_CUR_LIAB_AMT|LONG_TERM_LN_AMT|OTH_LONG_TERM_LIAB_NM|OTH_LONG_TERM_LIAB_A

In [88]:
#  构建 SQL 语句
sql_query_insrt_iib = f"""INSERT INTO bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
    (       
            RTN_RESP_EV_ID              ,
            AI_ENT_CD                   ,
            RPT_POS_DT                  ,
            SBMN_DTTM                   ,
            HALF_YR_END_DT              ,
            APPR_MONY_BRKR_NM           ,
            TNGBL_ASSET_AMT             ,
            INV_AMT                     ,
            OTH_NON_CUR_ASSET_NM        ,
            OTH_NON_CUR_ASSET_AMT       ,
            TOT_NON_CUR_ASSET_AMT       ,
            CASH_AT_BANK_AND_IN_HAND_AMT,
            DEBTOR_AMT                  ,
            OTH_CUR_ASSET_NM            ,  
            OTH_CUR_ASSET_AMT           , 
            TOT_CUR_ASSET_AMT           ,  
            TOT_ASSET_AMT               ,
            CROR_AMT                    ,
            LN_AMT                      ,
            DFR_TAX_AMT                 ,
            OTH_CUR_LIAB_NM             ,  
            OTH_CUR_LIAB_AMT            ,  
            TOT_CUR_LIAB_AMT            ,  
            LONG_TERM_LN_AMT            ,  
            OTH_LONG_TERM_LIAB_NM       ,
            OTH_LONG_TERM_LIAB_AMT      ,
            TOT_LONG_TERM_LIAB_AMT      ,
            TOT_LIAB_AMT                ,
            NET_ASSET_AMT               ,
            PD_UP_SHR_CAP_AMT           ,
            SHR_PREM_ACCT_AMT           ,
            REVALQ_RESV_AMT             ,
            OTH_RESV_NM                 ,
            OTH_RESV_AMT                ,
            PRFT_AND_LOSS_ACCT_AMT      ,
            TOT_SHRHLD_FUND_AMT         ,
            ROW_VLD_STS_CD              ,
            ROW_VLD_MSG_TXT             ,
            TX_DT                       ,
            INSE_DTTM                   ,
            UPDT_DTTM
    )
SELECT
    sha2(CONCAT(
        COALESCE(AI_ENT_CD, ''),
        COALESCE(CAST(TO_DATE(RPT_POS_DT, 'yyyyMMdd') AS STRING), ''),
        COALESCE(CAST(TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss') AS STRING), '')
    ),256)                                                                                 AS RTN_RESP_EV_ID,
    CAST(AI_ENT_CD AS STRING)                                                              AS AI_ENT_CD,
    TO_DATE(RPT_POS_DT, 'yyyyMMdd')                                                        AS RPT_POS_DT,
    TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss')                                              AS SBMN_DTTM,
    CAST(SUBSTRING(TRIM(HALF_YR_END_DT), 1, 10) AS DATE)                                   AS HALF_YR_END_DT,
    CAST(APPR_MONY_BRKR_NM AS STRING)                                                      AS APPR_MONY_BRKR_NM,
    CAST(REPLACE(CAST(TNGBL_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))              AS TNGBL_ASSET_AMT,
    CAST(REPLACE(CAST(INV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                      AS INV_AMT,
    CAST(OTH_NON_CUR_ASSET_NM AS STRING)                                                   AS OTH_NON_CUR_ASSET_NM,
    CAST(REPLACE(CAST(OTH_NON_CUR_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))        AS OTH_NON_CUR_ASSET_AMT,
    CAST(REPLACE(CAST(TOT_NON_CUR_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))        AS TOT_NON_CUR_ASSET_AMT,
    CAST(REPLACE(CAST(CASH_AT_BANK_AND_IN_HAND_AMT AS STRING), ',' ,'') AS DECIMAL(30,10)) AS CASH_AT_BANK_AND_IN_HAND_AMT,
    CAST(REPLACE(CAST(DEBTOR_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                   AS DEBTOR_AMT,
    CAST(OTH_CUR_ASSET_NM AS STRING)                                                       AS OTH_CUR_ASSET_NM,
    CAST(REPLACE(CAST(OTH_CUR_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))            AS OTH_CUR_ASSET_AMT,
    CAST(REPLACE(CAST(TOT_CUR_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))            AS TOT_CUR_ASSET_AMT,
    CAST(REPLACE(CAST(TOT_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                AS TOT_ASSET_AMT,
    CAST(REPLACE(CAST(CROR_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                     AS CROR_AMT,
    CAST(REPLACE(CAST(LN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                       AS LN_AMT,
    CAST(REPLACE(CAST(DFR_TAX_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                  AS DFR_TAX_AMT,
    CAST(OTH_CUR_LIAB_NM AS STRING)                                                        AS OTH_CUR_LIAB_NM,
    CAST(REPLACE(CAST(OTH_CUR_LIAB_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))             AS OTH_CUR_LIAB_AMT,
    CAST(REPLACE(CAST(TOT_CUR_LIAB_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))             AS TOT_CUR_LIAB_AMT,
    CAST(REPLACE(CAST(LONG_TERM_LN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))             AS LONG_TERM_LN_AMT,
    CAST(OTH_LONG_TERM_LIAB_NM AS STRING)                                                  AS OTH_LONG_TERM_LIAB_NM,
    CAST(REPLACE(CAST(OTH_LONG_TERM_LIAB_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))       AS OTH_LONG_TERM_LIAB_AMT,
    CAST(REPLACE(CAST(TOT_LONG_TERM_LIAB_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))       AS TOT_LONG_TERM_LIAB_AMT,
    CAST(REPLACE(CAST(TOT_LIAB_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                 AS TOT_LIAB_AMT,
    CAST(REPLACE(CAST(NET_ASSET_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                AS NET_ASSET_AMT,
    CAST(REPLACE(CAST(PD_UP_SHR_CAP_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))            AS PD_UP_SHR_CAP_AMT,
    CAST(REPLACE(CAST(SHR_PREM_ACCT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))            AS SHR_PREM_ACCT_AMT,
    CAST(REPLACE(CAST(REVALQ_RESV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))              AS REVALQ_RESV_AMT,
    CAST(OTH_RESV_NM AS STRING)                                                            AS OTH_RESV_NM,
    CAST(REPLACE(CAST(OTH_RESV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                 AS OTH_RESV_AMT,
    CAST(REPLACE(CAST(PRFT_AND_LOSS_ACCT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))       AS PRFT_AND_LOSS_ACCT_AMT,
    CAST(REPLACE(CAST(TOT_SHRHLD_FUND_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))          AS TOT_SHRHLD_FUND_AMT,
    CAST(ROW_VLD_STS_CD AS STRING)                                                         AS ROW_VLD_STS_CD,
    CAST(ROW_VLD_MSG_TXT AS STRING)                                                        AS ROW_VLD_MSG_TXT,
    TO_DATE(CAST('{{tx_dt}}' AS STRING), 'yyyyMMdd')                                       AS TX_DT,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                              AS INSE_DTTM,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                              AS UPDT_DTTM
FROM temp_view_amb_part_iib_fin_pos_as_at_the_end_of_the_half t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
"""

In [89]:
#  构建 SQL 语句
sql_query_insrt_iv = f"""INSERT INTO bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
    (    
        RTN_RESP_EV_ID                                                    ,
        AI_ENT_CD                                                          ,
        RPT_POS_DT                                                         ,
        SBMN_DTTM                                                          ,
        HALF_YR_END_DT                                                     ,
        APPR_MONY_BRKR_NM                                                  ,
        TRAN_BSS_REVN_GEN_IND                                              ,
        TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                ,
        TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                          ,
        TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                               ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                         ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                        ,
        FX_DRTV_FX_SWAP_AMT                                                ,
        FX_DRTV_NDF_AMT                                                    ,
        FX_DRTV_NDO_AMT                                                    ,
        FX_DRTV_VANILLA_FX_OPT_AMT                                         ,
        FX_DRTV_OTH_FX_DRTV_NM                                             ,
        FX_DRTV_OTH_FX_DRTV_AMT                                            ,
        INT_RT_DRTV_SNGL_CURY_IRS_AMT                                      ,
        INT_RT_DRTV_CROSS_CURY_IRS_AMT                                     ,
        INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                   ,
        INT_RT_DRTV_FRA_AMT                                                ,
        INT_RT_DRTV_INT_RT_OPT_AMT                                         ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                     ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                    ,
        OTH_MONY_MKT_OTC_DRTV_I_NM                                         ,
        OTH_MONY_MKT_OTC_DRTV_I_AMT                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_NM                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_AMT                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_NM                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_AMT                                      ,
        BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                          ,
        BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN       ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                               ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                              ,
        TOT_HK_RELVNT_BUSN_TRAN_AMT                                        ,
        ROW_VLD_STS_CD                                                     ,
        ROW_VLD_MSG_TXT                                                    ,
        TX_DT                                                              ,
        INSE_DTTM                                                          ,
        UPDT_DTTM
    )
SELECT
    sha2(CONCAT(
        COALESCE(AI_ENT_CD, ''),
        COALESCE(CAST(TO_DATE(RPT_POS_DT, 'yyyyMMdd') AS STRING), ''),
        COALESCE(CAST(TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss') AS STRING), '')
    ),256)                                                                                                                      AS RTN_RESP_EV_ID,
    CAST(AI_ENT_CD AS STRING)                                                                                                   AS AI_ENT_CD,
    TO_DATE(RPT_POS_DT, 'yyyyMMdd')                                                                                             AS RPT_POS_DT,
    TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss')                                                                                   AS SBMN_DTTM,
    CAST(SUBSTRING(TRIM(HALF_YR_END_DT), 1, 10) AS DATE)                                                                        AS HALF_YR_END_DT,
    CAST(APPR_MONY_BRKR_NM AS STRING)                                                                                           AS APPR_MONY_BRKR_NM,
    CAST(TRAN_BSS_REVN_GEN_IND AS STRING)                                                                                       AS TRAN_BSS_REVN_GEN_IND,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                               AS TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                              AS TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT,
    CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM AS STRING)                                                                  AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                       AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT,      
    CAST(REPLACE(CAST(FX_DRTV_FX_SWAP_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS FX_DRTV_FX_SWAP_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDF_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDF_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDO_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDO_AMT,
    CAST(REPLACE(CAST(FX_DRTV_VANILLA_FX_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS FX_DRTV_VANILLA_FX_OPT_AMT,
    CAST(FX_DRTV_OTH_FX_DRTV_NM AS STRING)                                                                                      AS FX_DRTV_OTH_FX_DRTV_NM,       
    CAST(REPLACE(CAST(FX_DRTV_OTH_FX_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                           AS FX_DRTV_OTH_FX_DRTV_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_SNGL_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS INT_RT_DRTV_SNGL_CURY_IRS_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_CROSS_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                    AS INT_RT_DRTV_CROSS_CURY_IRS_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                  AS INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_FRA_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS INT_RT_DRTV_FRA_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS INT_RT_DRTV_INT_RT_OPT_AMT,
    CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_NM AS STRING)                                                                              AS INT_RT_DRTV_OTH_INT_RT_DRTV_NM,
    CAST(REPLACE(CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                   AS INT_RT_DRTV_OTH_INT_RT_DRTV_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_I_NM AS STRING)                                                                                  AS OTH_MONY_MKT_OTC_DRTV_I_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_I_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS OTH_MONY_MKT_OTC_DRTV_I_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_II_NM AS STRING)                                                                                 AS OTH_MONY_MKT_OTC_DRTV_II_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_II_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                      AS OTH_MONY_MKT_OTC_DRTV_II_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_III_NM AS STRING)                                                                                AS OTH_MONY_MKT_OTC_DRTV_III_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_III_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS OTH_MONY_MKT_OTC_DRTV_III_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN AS STRING), ',' ,'') AS DECIMAL(30,10))      AS BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN,
    CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM AS STRING)                                                                        AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                             AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT,
    CAST(REPLACE(CAST(TOT_HK_RELVNT_BUSN_TRAN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS TOT_HK_RELVNT_BUSN_TRAN_AMT,
    CAST(ROW_VLD_STS_CD AS STRING)                                                                                              AS ROW_VLD_STS_CD,   
    CAST(ROW_VLD_MSG_TXT AS STRING)                                                                                             AS ROW_VLD_MSG_TXT,  
    TO_DATE(CAST('{{tx_dt}}' AS STRING), 'yyyyMMdd')                                                                            AS TX_DT,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS INSE_DTTM,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS UPDT_DTTM
FROM temp_view_amb_part_iv_revn_from_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);

"""

In [90]:
#  构建 SQL 语句
sql_query_insrt_iii = f"""INSERT INTO bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
    (    
        SURV_RESP_EV_ID                                                    ,
        AI_ENT_CD                                                          ,
        RPT_POS_DT                                                         ,
        SBMN_DTTM                                                          ,
        HALF_YR_END_DT                                                     ,
        APPR_MONY_BRKR_NM                                                  ,
        TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                ,
        TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                          ,
        TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                               ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                         ,
        TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                        ,
        FX_DRTV_FX_SWAP_AMT                                                ,
        FX_DRTV_NDF_AMT                                                    ,
        FX_DRTV_NDO_AMT                                                    ,
        FX_DRTV_VANILLA_FX_OPT_AMT                                         ,
        FX_DRTV_OTH_FX_DRTV_NM                                             ,
        FX_DRTV_OTH_FX_DRTV_AMT                                            ,
        INT_RT_DRTV_SNGL_CURY_IRS_AMT                                      ,
        INT_RT_DRTV_CROSS_CURY_IRS_AMT                                     ,
        INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                   ,
        INT_RT_DRTV_FRA_AMT                                                ,
        INT_RT_DRTV_INT_RT_OPT_AMT                                         ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                     ,
        INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                    ,
        OTH_MONY_MKT_OTC_DRTV_I_NM                                         ,
        OTH_MONY_MKT_OTC_DRTV_I_AMT                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_NM                                        ,
        OTH_MONY_MKT_OTC_DRTV_II_AMT                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_NM                                       ,
        OTH_MONY_MKT_OTC_DRTV_III_AMT                                      ,
        BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                          ,
        BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN       ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                               ,
        BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                              ,
        TOT_HK_RELVNT_BUSN_TRAN_AMT                                        ,
        ROW_VLD_STS_CD                                                     ,
        ROW_VLD_MSG_TXT                                                    ,
        TX_DT                                                              ,
        INSE_DTTM                                                          ,
        UPDT_DTTM
    )
SELECT
    sha2(CONCAT(
        COALESCE(AI_ENT_CD, ''),
        COALESCE(CAST(TO_DATE(RPT_POS_DT, 'yyyyMMdd') AS STRING), ''),
        COALESCE(CAST(TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss') AS STRING), '')
    ),256)                                                                                                                      AS SURV_RESP_EV_ID,
    CAST(AI_ENT_CD AS STRING)                                                                                                   AS AI_ENT_CD,
    TO_DATE(RPT_POS_DT, 'yyyyMMdd')                                                                                             AS RPT_POS_DT,
    TO_TIMESTAMP(SBMN_DTTM, 'yyyyMMddHHmmss')                                                                                   AS SBMN_DTTM,
    CAST(SUBSTRING(TRIM(HALF_YR_END_DT), 1, 10) AS DATE)                                                                        AS HALF_YR_END_DT,
    CAST(APPR_MONY_BRKR_NM AS STRING)                                                                                           AS APPR_MONY_BRKR_NM,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                               AS TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                              AS TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT,
    CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM AS STRING)                                                                  AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM,
    CAST(REPLACE(CAST(TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                       AS TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT,      
    CAST(REPLACE(CAST(FX_DRTV_FX_SWAP_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS FX_DRTV_FX_SWAP_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDF_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDF_AMT,
    CAST(REPLACE(CAST(FX_DRTV_NDO_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                                   AS FX_DRTV_NDO_AMT,
    CAST(REPLACE(CAST(FX_DRTV_VANILLA_FX_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS FX_DRTV_VANILLA_FX_OPT_AMT,
    CAST(FX_DRTV_OTH_FX_DRTV_NM AS STRING)                                                                                      AS FX_DRTV_OTH_FX_DRTV_NM,       
    CAST(REPLACE(CAST(FX_DRTV_OTH_FX_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                           AS FX_DRTV_OTH_FX_DRTV_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_SNGL_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS INT_RT_DRTV_SNGL_CURY_IRS_AMT,   
    CAST(REPLACE(CAST(INT_RT_DRTV_CROSS_CURY_IRS_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                    AS INT_RT_DRTV_CROSS_CURY_IRS_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                  AS INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_FRA_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                               AS INT_RT_DRTV_FRA_AMT,
    CAST(REPLACE(CAST(INT_RT_DRTV_INT_RT_OPT_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                        AS INT_RT_DRTV_INT_RT_OPT_AMT,
    CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_NM AS STRING)                                                                              AS INT_RT_DRTV_OTH_INT_RT_DRTV_NM,
    CAST(REPLACE(CAST(INT_RT_DRTV_OTH_INT_RT_DRTV_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                   AS INT_RT_DRTV_OTH_INT_RT_DRTV_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_I_NM AS STRING)                                                                                  AS OTH_MONY_MKT_OTC_DRTV_I_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_I_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS OTH_MONY_MKT_OTC_DRTV_I_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_II_NM AS STRING)                                                                                 AS OTH_MONY_MKT_OTC_DRTV_II_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_II_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                      AS OTH_MONY_MKT_OTC_DRTV_II_AMT,
    CAST(OTH_MONY_MKT_OTC_DRTV_III_NM AS STRING)                                                                                AS OTH_MONY_MKT_OTC_DRTV_III_NM,
    CAST(REPLACE(CAST(OTH_MONY_MKT_OTC_DRTV_III_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                     AS OTH_MONY_MKT_OTC_DRTV_III_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                         AS BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN AS STRING), ',' ,'') AS DECIMAL(30,10))      AS BUSN_REL_ON_EXPT_UND_SFO_PROVDE_CLNT_CLRG_SRVC_FOR_OTCD_TRAN,
    CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM AS STRING)                                                                        AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM,
    CAST(REPLACE(CAST(BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                             AS BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT,
    CAST(REPLACE(CAST(TOT_HK_RELVNT_BUSN_TRAN_AMT AS STRING), ',' ,'') AS DECIMAL(30,10))                                       AS TOT_HK_RELVNT_BUSN_TRAN_AMT,
    CAST(ROW_VLD_STS_CD AS STRING)                                                                                              AS ROW_VLD_STS_CD,   
    CAST(ROW_VLD_MSG_TXT AS STRING)                                                                                             AS ROW_VLD_MSG_TXT,  
    TO_DATE(CAST('{{tx_dt}}' AS STRING), 'yyyyMMdd')                                                                            AS TX_DT,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS INSE_DTTM,
    FROM_UTC_TIMESTAMP(CURRENT_TIMESTAMP(), 'Asia/Hong_Kong')                                                                   AS UPDT_DTTM
FROM temp_view_amb_part_iii_tran_amount_of_hk_relvnt_busn t1
WHERE NOT EXISTS (
    SELECT 1
    FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN t2
    WHERE t2.AI_ENT_CD = t1.AI_ENT_CD
        AND t2.RPT_POS_DT = TO_DATE(t1.RPT_POS_DT,'yyyyMMdd')
        AND t2.SBMN_DTTM > TO_TIMESTAMP(t1.SBMN_DTTM,'yyyyMMddHHmmss')
);
"""

In [91]:

#  执行 SQL 语句
spark.sql(sql_query_insrt_iib)
spark.sql(sql_query_insrt_iv)
spark.sql(sql_query_insrt_iii)
print("All tables Inser done!")

All tables Inser done!


In [92]:
#check data after insert:
print("check data after insert:")
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN").show()
print("AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")
spark.sql("select * from bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").show()

check data after insert:
AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------------+---------+----------+-------------------+--------------+-----------------+---------------+------------+--------------------+---------------------+---------------------+----------------------------+-------------+----------------+-----------------+-----------------+-------------+-------------+-------------+-------------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+--------------+---------------+-----------------+-----------------+---------------+-----------+-------------+----------------------+-------------------+--------------------+--------------------+-----+--------------------+--------------------+
|      RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|          SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|     INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_I

In [93]:
#Data Validation: AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
print("Data Validation for AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("""
SELECT * 
FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 
ORDER BY RTN_RESP_EV_ID, APPR_MONY_BRKR_NM DESC;""").show()

spark.sql("""
SELECT COUNT(*) 
FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF;""").show()

spark.sql("""
SELECT 1 AS DUP_CHK, RTN_RESP_EV_ID, APPR_MONY_BRKR_NM, COUNT(*) AS NUMS
FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
GROUP BY RTN_RESP_EV_ID, APPR_MONY_BRKR_NM
HAVING COUNT(*) > 1;""").show()

spark.sql("""
SELECT 2 AS NULL_CHK, * 
FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
WHERE RTN_RESP_EV_ID IS NULL OR APPR_MONY_BRKR_NM IS NULL;""").show()

spark.sql("""
SELECT SUM(TNGBL_ASSET_AMT) AS TNGBL_ASSET_AMT
FROM bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF;
""").show()

Data Validation for AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
+--------------------+---------+----------+-------------------+--------------+-----------------+---------------+------------+--------------------+---------------------+---------------------+----------------------------+-------------+----------------+-----------------+-----------------+-------------+-------------+-------------+-------------+---------------+----------------+----------------+----------------+---------------------+----------------------+----------------------+--------------+---------------+-----------------+-----------------+---------------+-----------+-------------+----------------------+-------------------+--------------------+--------------------+-----+--------------------+--------------------+
|      RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|          SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|TNGBL_ASSET_AMT|     INV_AMT|OTH_NON_CUR_ASSET_NM|OTH_NON_CUR_ASSET_AMT|TOT_NON_CUR_ASSET_AMT|CASH_AT_BANK_AND_IN_HAN

In [94]:
#Data Validation: AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
print("Data Validation for AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("""
SELECT * 
FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 
ORDER BY RTN_RESP_EV_ID, APPR_MONY_BRKR_NM DESC;""").show()

spark.sql("""
SELECT COUNT(*) 
FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN;""").show()

spark.sql("""
SELECT 1 AS DUP_CHK, RTN_RESP_EV_ID, APPR_MONY_BRKR_NM, COUNT(*) AS NUMS
FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
GROUP BY RTN_RESP_EV_ID, APPR_MONY_BRKR_NM
HAVING COUNT(*) > 1;""").show()

spark.sql("""
SELECT 2 AS NULL_CHK, * 
FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
WHERE RTN_RESP_EV_ID IS NULL OR APPR_MONY_BRKR_NM IS NULL;""").show()

spark.sql("""
SELECT SUM(FX_DRTV_NDF_AMT) AS SUM_FX_DRTV_NDF_AMT
FROM bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN;""").show()


Data Validation for AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
+--------------------+---------+----------+-------------------+--------------+-----------------+---------------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+--------------------+--------------------+--------------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+--------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+-------

In [95]:
#Data Validation: AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
print("Data Validation for AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

spark.sql("""
SELECT * 
FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN 
ORDER BY SURV_RESP_EV_ID, APPR_MONY_BRKR_NM DESC;""").show()

spark.sql("""
SELECT COUNT(*) 
FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN;""").show()

spark.sql("""
SELECT 1 AS DUP_CHK, SURV_RESP_EV_ID, APPR_MONY_BRKR_NM, COUNT(*) AS NUMS
FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
GROUP BY SURV_RESP_EV_ID, APPR_MONY_BRKR_NM
HAVING COUNT(*) > 1;""").show()

spark.sql("""
SELECT 2 AS NULL_CHK, * 
FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
WHERE SURV_RESP_EV_ID IS NULL OR APPR_MONY_BRKR_NM IS NULL;""").show()

spark.sql("""
SELECT SUM(TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT) AS SUM_TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT
FROM bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN;""").show()

Data Validation for AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN
+--------------------+---------+----------+-------------------+--------------+-----------------+-----------------------------------+-----------------------------------------+------------------------------------+------------------------------------------+-------------------------------------------+-------------------+---------------+---------------+--------------------------+----------------------+-----------------------+-----------------------------+------------------------------+--------------------------------+-------------------+--------------------------+------------------------------+-------------------------------+--------------------------+---------------------------+---------------------------+----------------------------+----------------------------+-----------------------------+-----------------------------------------+------------------------------------------------------------+------------------------------------+-

In [96]:
# 方式 A：运行 SQL
# spark.sql("TRUNCATE TABLE bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")

# 方式 B：如果你想通过覆盖空 DataFrame 的方式（不推荐，TRUNCATE 更专业）
# spark.table("bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN").limit(0).write.mode("overwrite").saveAsTable("...")

In [99]:

# 1. 读取表数据到 DataFrame
df_iib = spark.table("bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
df_iv = spark.table("bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
df_iii = spark.table("bc2_cde_rstk.AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN")


# 2. 转换为 Pandas 并保存（路径建议选在你的 ldp 项目下）
output_path_iib = "/home/phil/ldp/csv_output/AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF.csv"
output_path_iv = "/home/phil/ldp/csv_output/AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN.csv"
output_path_iii = "/home/phil/ldp/csv_output/AMB_PART_III_TRAN_AMT_OF_HK_RELVNT_BUSN.csv"

df_iib.toPandas().to_csv(output_path_iib, index=False, encoding='utf-8-sig')
df_iv.toPandas().to_csv(output_path_iv, index=False, encoding='utf-8-sig')
df_iii.toPandas().to_csv(output_path_iii, index=False, encoding='utf-8-sig')


print(f"✅ 文件已导出至: /home/phil/ldp/csv_output/")

✅ 文件已导出至: /home/phil/ldp/csv_output/
